# 09 Autoformer NeuralForecast

This notebook evaluates NeuralForecast's Autoformer on fixed B.

The first part keeps the previous fixed setting. The second part runs a small manual grid search to check whether the weak fixed-setting result is caused by the model family or by a single poor configuration.

This remains a reference deep-learning baseline, not the main analysis model. The model is univariate, uses no exogenous variables, trains on `y = log(number_parcels)`, and is evaluated on the original `number_parcels` scale.

Note: On Windows + Python 3.13, `ray` may not be installable. NeuralForecast imports `ray` for auto-tuning classes even when fixed hyperparameters are used. The helper module installs minimal in-memory stubs for those auto-tuning imports. This notebook does not use ray or NeuralForecast's automatic hyperparameter tuning.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "Transport_amount_project" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_connected_parcel_data
from src.forecasting.autoformer_neuralforecast import (
    GRID_SPEC_NAME,
    MODEL_NAME,
    SPEC_NAME,
    fit_autoformer_neuralforecast,
    forecast_autoformer_neuralforecast,
    run_autoformer_neuralforecast_grid_search,
)
from src.forecasting.evaluation import evaluate_forecasts
from src.forecasting.splits import make_fixed_split_b

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "parcel_volume_connected.csv"
FORECAST_DIR = PROJECT_ROOT / "output" / "forecasts"
PREDICTIONS_DIR = FORECAST_DIR / "predictions"
METRICS_DIR = FORECAST_DIR / "metrics"
FIGURES_DIR = FORECAST_DIR / "figures"

for path in [PREDICTIONS_DIR, METRICS_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Model:", MODEL_NAME)
print("Fixed spec:", SPEC_NAME)
print("Grid spec:", GRID_SPEC_NAME)

Project root: C:\Users\fugat\Desktop\python_project\Transport_amount_project
Model: autoformer_neuralforecast
Fixed spec: neuralforecast_autoformer_minimal
Grid spec: neuralforecast_autoformer_grid


## 1. Load Data And Build fixed B

The connected monthly parcel series is loaded from `data/processed/parcel_volume_connected.csv`. fixed B trains on 2002-04 to 2023-12 and forecasts 2024-01 to 2026-02.

In [2]:
df = load_connected_parcel_data(str(DATA_PATH))
if "y" not in df.columns:
    df["y"] = np.log(df["number_parcels"].astype(float))

split = make_fixed_split_b(df)
train = split["train"]
test = split["test"]

split_summary = pd.DataFrame(
    [
        {
            "split": split["split"],
            "train_start": train.index.min().date(),
            "train_end": train.index.max().date(),
            "train_rows": len(train),
            "test_start": test.index.min().date(),
            "test_end": test.index.max().date(),
            "test_rows": len(test),
        }
    ]
)
display(split_summary)

,split,train_start,train_end,train_rows,test_start,test_end,test_rows
0,fixed_b,2002-04-01,2023-12-01,261,2024-01-01,2026-02-01,26


## 2. Fixed Setting Baseline

This reproduces the minimal fixed-setting NeuralForecast Autoformer. The moving-average window is set to 13 because the Autoformer decomposition uses a centered moving average and an odd window avoids length mismatch errors.

In [3]:
model_bundle = fit_autoformer_neuralforecast(
    train,
    target_col="y",
    horizon=len(test),
    input_size=24,
    hidden_size=16,
    n_head=2,
    encoder_layers=1,
    decoder_layers=1,
    conv_hidden_size=16,
    moving_avg_window=13,
    max_steps=200,
    random_seed=42,
    spec_name=SPEC_NAME,
)

fit_info = {
    **model_bundle.config,
    **{f"version_{key}": value for key, value in model_bundle.versions.items()},
    "train_seconds": model_bundle.train_seconds,
}
fit_summary = pd.DataFrame([fit_info])
forecast_df = forecast_autoformer_neuralforecast(model_bundle, test, split="fixed_b", spec_name=SPEC_NAME)
metrics_df = evaluate_forecasts(forecast_df, y_train=train["number_parcels"])

display(fit_summary.T.rename(columns={0: "value"}))
display(metrics_df)

Seed set to 42


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


C:\Users\fugat\Desktop\python_project\Transport_amount_project\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


C:\Users\fugat\Desktop\python_project\Transport_amount_project\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


,value
model,autoformer_neuralforecast
spec_name,neuralforecast_autoformer_minimal
target_col,y
target_scale_training,log
horizon,26
input_size,24
hidden_size,16
n_head,2
encoder_layers,1
decoder_layers,1


,model,split,forecast_type,spec_name,n,rmse,mae,mape,mase
0,autoformer_neuralforecast,fixed_b,unconditional,neuralforecast_autoformer_minimal,26,32929.373359,24639.602782,6.262312,1.970093


## 3. Small Manual Grid Search

This is a small manual search, not full deep-learning hyperparameter tuning. It tests 12 configurations only:

- `input_size`: 12, 24, 36
- `hidden_size`: 16, 32
- `max_steps`: 100, 200

The model remains fixed B only, univariate, and unconditional. Failed configurations are kept in the grid result with `status="failed"` and an error message.

In [4]:
grid_configs = [
    {
        "input_size": input_size,
        "hidden_size": hidden_size,
        "max_steps": max_steps,
        "n_head": 2,
        "encoder_layers": 1,
        "decoder_layers": 1,
        "conv_hidden_size": hidden_size,
        "moving_avg_window": 13,
        "random_seed": 42,
    }
    for input_size in [12, 24, 36]
    for hidden_size in [16, 32]
    for max_steps in [100, 200]
]

print("configs:", len(grid_configs))
display(pd.DataFrame(grid_configs))

configs: 12


,input_size,hidden_size,max_steps,n_head,encoder_layers,decoder_layers,conv_hidden_size,moving_avg_window,random_seed
0,12,16,100,2,1,1,16,13,42
1,12,16,200,2,1,1,16,13,42
2,12,32,100,2,1,1,32,13,42
3,12,32,200,2,1,1,32,13,42
4,24,16,100,2,1,1,16,13,42
5,24,16,200,2,1,1,16,13,42
6,24,32,100,2,1,1,32,13,42
7,24,32,200,2,1,1,32,13,42
8,36,16,100,2,1,1,16,13,42
9,36,16,200,2,1,1,16,13,42


In [5]:
grid_df, best_forecast_df, best_metrics_df, best_bundle = run_autoformer_neuralforecast_grid_search(
    train=train,
    test=test,
    configs=grid_configs,
    target_col="y",
    split="fixed_b",
)

grid_df = grid_df.sort_values(["status", "rmse", "config_id"], na_position="last").reset_index(drop=True)
display(grid_df)

success_count = int((grid_df["status"] == "success").sum())
failed_count = int((grid_df["status"] == "failed").sum())
print("success:", success_count, "failed:", failed_count)

Seed set to 42


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


C:\Users\fugat\Desktop\python_project\Transport_amount_project\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


GPU available: False, used: False


...

GPU available: False, used: False


TPU available: False, using: 0 TPU cores


C:\Users\fugat\Desktop\python_project\Transport_amount_project\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


,config_id,model,split,forecast_type,information_set,spec_name,target_col,status,error,input_size,...,training_time_seconds,total_time_seconds,rmse,mae,mape,mase,version_neuralforecast,version_pytorch_lightning,version_torch,version_ray_stub_used
0,8,autoformer_neuralforecast,fixed_b,unconditional,unconditional,neuralforecast_autoformer_grid,y,success,,24,...,6.393471,6.456061,21538.483310,17628.141837,4.605024,1.409482,3.1.9,2.5.6,2.12.1+cpu,True
1,7,autoformer_neuralforecast,fixed_b,unconditional,unconditional,neuralforecast_autoformer_grid,y,success,,24,...,4.510369,4.578971,24499.125451,20538.443895,5.329754,1.642179,3.1.9,2.5.6,2.12.1+cpu,True
2,10,autoformer_neuralforecast,fixed_b,unconditional,unconditional,neuralforecast_autoformer_grid,y,success,,36,...,4.878859,4.940613,29776.912217,25738.926873,6.756767,2.057991,3.1.9,2.5.6,2.12.1+cpu,True
3,9,autoformer_neuralforecast,fixed_b,unconditional,unconditional,neuralforecast_autoformer_grid,y,success,,36,...,3.974625,4.051129,31034.886301,26192.461349,6.877185,2.094254,3.1.9,2.5.6,2.12.1+cpu,True
4,11,autoformer_neuralforecast,fixed_b,unconditional,unconditional,neuralforecast_autoformer_grid,y,success,,36,...,4.859222,4.907565,32781.942302,25466.477986,6.751076,2.036207,3.1.9,2.5.6,2.12.1+cpu,True
5,5,autoformer_neuralforecast,fixed_b,unconditional,unconditional,neuralforecast_autoformer_grid,y,success,,24,...,2.360979,2.431222,32929.373359,24639.602782,6.262312,1.970093,3.1.9,2.5.6,2.12.1+cpu,True
6,6,autoformer_neuralforecast,fixed_b,unconditional,unconditional,neuralforecast_autoformer_grid,y,success,,24,...,2.253965,2.313945,32929.373359,24639.602782,6.262312,1.970093,3.1.9,2.5.6,2.12.1+cpu,True
7,12,autoformer_neuralforecast,fixed_b,unconditional,unconditional,neuralforecast_autoformer_grid,y,success,,36,...,7.169314,7.246723,33025.812136,25416.671526,6.747553,2.032224,3.1.9,2.5.6,2.12.1+cpu,True
8,3,autoformer_neuralforecast,fixed_b,unconditional,unconditional,neuralforecast_autoformer_grid,y,success,,12,...,3.258251,3.323130,34462.901106,24094.164482,5.863463,1.926482,3.1.9,2.5.6,2.12.1+cpu,True
9,4,autoformer_neuralforecast,fixed_b,unconditional,unconditional,neuralforecast_autoformer_grid,y,success,,12,...,3.050509,3.114940,34462.901106,24094.164482,5.863463,1.926482,3.1.9,2.5.6,2.12.1+cpu,True


success: 12 failed: 0


## 4. Save Grid Results And Best Model Outputs

The best configuration is selected by the smallest RMSE among successful configurations. If all configurations fail, only the grid-search CSV is saved and the notebook raises an error.

In [6]:
grid_path = METRICS_DIR / "autoformer_neuralforecast_grid_search.csv"
best_model_path = METRICS_DIR / "autoformer_neuralforecast_best_model.csv"
best_prediction_path = PREDICTIONS_DIR / "fixed_b_autoformer_neuralforecast_best.csv"
best_metrics_path = METRICS_DIR / "autoformer_neuralforecast_best_metrics.csv"
best_figure_path = FIGURES_DIR / "fixed_b_autoformer_neuralforecast_best_forecast.png"

grid_df.to_csv(grid_path, index=False)

if best_forecast_df is None or best_metrics_df is None or best_bundle is None:
    raise RuntimeError("All NeuralForecast Autoformer grid configurations failed. See grid search CSV.")

best_row = grid_df[grid_df["status"] == "success"].sort_values("rmse").head(1).copy()
best_row.to_csv(best_model_path, index=False)
best_forecast_df.to_csv(best_prediction_path, index=False)
best_metrics_df.to_csv(best_metrics_path, index=False)

fig, ax = plt.subplots(figsize=(10.5, 5.5))
train_tail = train.tail(24)
ax.plot(train_tail.index, train_tail["number_parcels"], color="0.55", linewidth=1.2, label="train actual tail")
ax.plot(test.index, test["number_parcels"], color="black", linewidth=1.6, label="test actual")
ax.plot(best_forecast_df["date"], best_forecast_df["y_pred"], marker="o", linewidth=1.2, label="autoformer_neuralforecast_best")
ax.axvline(train.index.max(), color="0.2", linestyle=":", linewidth=1.0)
ax.set_title("fixed_b: NeuralForecast Autoformer best grid forecast")
ax.set_xlabel("Date")
ax.set_ylabel("number_parcels")
ax.grid(True, color="0.85", linewidth=0.8)
ax.legend()
fig.tight_layout()
fig.savefig(best_figure_path, dpi=300, bbox_inches="tight")
plt.close(fig)

print("saved", grid_path)
print("saved", best_model_path)
print("saved", best_prediction_path)
print("saved", best_metrics_path)
print("saved", best_figure_path)
display(best_row.T.rename(columns={best_row.index[0]: "best"}))

saved C:\Users\fugat\Desktop\python_project\Transport_amount_project\output\forecasts\metrics\autoformer_neuralforecast_grid_search.csv
saved C:\Users\fugat\Desktop\python_project\Transport_amount_project\output\forecasts\metrics\autoformer_neuralforecast_best_model.csv
saved C:\Users\fugat\Desktop\python_project\Transport_amount_project\output\forecasts\predictions\fixed_b_autoformer_neuralforecast_best.csv
saved C:\Users\fugat\Desktop\python_project\Transport_amount_project\output\forecasts\metrics\autoformer_neuralforecast_best_metrics.csv
saved C:\Users\fugat\Desktop\python_project\Transport_amount_project\output\forecasts\figures\fixed_b_autoformer_neuralforecast_best_forecast.png


,best
config_id,8
model,autoformer_neuralforecast
split,fixed_b
forecast_type,unconditional
information_set,unconditional
spec_name,neuralforecast_autoformer_grid
target_col,y
status,success
error,
input_size,24


## 5. Compare With Existing fixed B Benchmarks

This comparison is only for orientation. `autoformer_neuralforecast` is an unconditional deep-learning baseline. It should mainly be compared with other unconditional models such as `autoformer_lite`, `seasonal_naive`, and Prophet without regressors.

In [7]:
comparison_rows = []
source_files = [
    "autoformer_lite_metrics.csv",
    "autoformer_neuralforecast_metrics.csv",
    "naive_metrics.csv",
    "prophet_metrics.csv",
]

for filename in source_files:
    path = METRICS_DIR / filename
    if not path.exists():
        print(f"skip: {filename} not found")
        continue
    frame = pd.read_csv(path)
    frame = frame[frame["split"] == "fixed_b"].copy()
    if filename == "naive_metrics.csv":
        frame = frame[frame["model"].isin(["seasonal_naive"])]
    if filename == "prophet_metrics.csv":
        frame = frame[frame["model"] == "prophet"]
    frame["source_file"] = filename
    comparison_rows.append(frame)

comparison_rows.append(best_metrics_df.assign(source_file="autoformer_neuralforecast_best_metrics.csv"))
comparison = pd.concat(comparison_rows, ignore_index=True, sort=False)
comparison = comparison[["model", "split", "forecast_type", "spec_name", "rmse", "mae", "mape", "mase", "source_file"]]
comparison = comparison.sort_values("rmse").reset_index(drop=True)
display(comparison)

,model,split,forecast_type,spec_name,rmse,mae,mape,mase,source_file
0,seasonal_naive,fixed_b,unconditional,post2020_m5,13622.783591,11619.923077,2.906761,0.929087,naive_metrics.csv
1,autoformer_neuralforecast,fixed_b,unconditional,neuralforecast_autoformer_grid,21538.483310,17628.141837,4.605024,1.409482,autoformer_neuralforecast_best_metrics.csv
2,prophet,fixed_b,unconditional,prophet_no_regressors,26730.003403,19845.066371,4.834269,1.586739,prophet_metrics.csv
3,autoformer_neuralforecast,fixed_b,unconditional,neuralforecast_autoformer_minimal,32929.373359,24639.602782,6.262312,1.970093,autoformer_neuralforecast_metrics.csv
4,autoformer_lite,fixed_b,unconditional,autoformer_lite_univariate,35587.389423,32410.636313,8.536476,2.591436,autoformer_lite_metrics.csv


## 6. Interpretation Notes

- This is a NeuralForecast library Autoformer baseline, not the earlier local `autoformer_lite` implementation.
- The grid is deliberately small and manual. It is not full deep-learning tuning.
- It is fixed B only, univariate, and unconditional. No event dummies or exogenous regressors are used.
- The dataset has only 287 monthly observations, so weak performance or overfitting is not surprising.
- `ray_stub_used=True` in the fit summary/grid output means the environment could not use real `ray`; this notebook uses fixed hyperparameters and does not call ray-based auto-tuning.